In [ ]:
# v2 — sorted/shuffled split toggle, multi-seed loop, Grad-CAM kept
!pip install --upgrade wandb -q


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import numpy as np, os, random, time, copy, gc
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import wandb

def _safe_auc(y_true, y_score, **kw):
    try:
        return roc_auc_score(y_true, y_score, **kw)
    except Exception as _e:
        print(f"[auc] guarded: {_e}")
        return 0.0

# ===================== EXPERIMENT CONTROL (added in v2) =====================
# Toggle these for each Kaggle run — everything else in the notebook is unchanged.
SPLIT_MODES      = ['shuffled', 'sorted']  # loop both; set to one to skip the other
SPLIT_RATIO      = (0.80, 0.20)    # (train, test) for matched 80/20
                                   # or (train, val, test) e.g. (0.70, 0.20, 0.10)
SEEDS            = [42, 123, 2024] # paper table; quick run: [42]
_SPLIT_MODE      = SPLIT_MODES[0]  # mutated inside the main loop
                                   # 'shuffled' (leaky baseline used by prior papers)

USE_WORKER_INIT  = True            # deterministic DataLoader workers
LUNG_CROP_THRESH = 60              # was 20; raise to drop CT gantry/black border
WANDB_ENABLED    = True            # keep W&B on; set False for a silent local run
WANDB_API_KEY    = None            # None = use ~/.netrc / Kaggle secret. Paste key to override.
WANDB_PROJECT    = 'lung-honest-bench'

import pandas as pd

def _winit(worker_id):
    import torch as _t, numpy as _n, random as _r
    s = _t.initial_seed() % 2**32
    _n.random.seed(s); _r.seed(s)

def _loader_kwargs(seed):
    if not USE_WORKER_INIT:
        return {}
    g = torch.Generator(); g.manual_seed(int(seed))
    return dict(worker_init_fn=_winit, generator=g)

def make_split(per_class_files, mode, ratio, seed):
    """
    per_class_files: dict {class_id: [(path, label), ...]}  (already sorted by filename)
    mode: 'sorted' keeps filename order; 'shuffled' randomly permutes within each class.
    ratio: (train, test) or (train, val, test)
    Returns: train_list, val_list (or []), test_list
    """
    rng = np.random.default_rng(int(seed))
    train, val, test = [], [], []
    for cid, items in per_class_files.items():
        items = list(items)
        if mode == 'shuffled':
            idx = np.arange(len(items)); rng.shuffle(idx); items = [items[i] for i in idx]
        n = len(items)
        if len(ratio) == 2:
            n_tr = int(n*ratio[0])
            train += items[:n_tr]; test += items[n_tr:]
        else:
            n_tr = int(n*ratio[0]); n_va = int(n*ratio[1])
            train += items[:n_tr]; val += items[n_tr:n_tr+n_va]; test += items[n_tr+n_va:]
    return train, val, test

def _wandb_login_safe():
    if not WANDB_ENABLED: return
    try:
        if WANDB_API_KEY:
            wandb.login(key=WANDB_API_KEY); return
        try:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret('wandb_api_key')); return
        except Exception:
            pass
        wandb.login()  # falls back to ~/.netrc
    except Exception as e:
        print(f'[wandb] login skipped: {e}')

def _wandb_init_safe(reinit=True, **kwargs):
    if not WANDB_ENABLED:
        # Use real wandb in disabled mode so wandb.log/wandb.Image/wandb.finish
        # remain valid no-ops elsewhere in the notebook.
        try:
            return wandb.init(mode='disabled', project=kwargs.get('project','disabled'),
                              name=kwargs.get('name','disabled'), reinit=True)
        except Exception:
            class _Noop:
                def log(self,*a,**k): pass
                def finish(self,*a,**k): pass
            return _Noop()
    return wandb.init(**kwargs)
# ============================================================================


# ===================== CONFIG =====================
# Lung-EffNet Paper: Raza et al. (2023)
# HONEST BENCHMARK: sequential 70/20/10 split (NO shuffle)
# Same architecture, same hyperparameters — only split changes
# ==================================================

config = {
    'data_dir': '/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset',
    'batch_size': 32,
    'learning_rate': 0.001,
    'epochs': 30,
    'dropout': 0.5,
    'lr_decay_factor': 0.3,
    'lr_patience': 5,
    'train_ratio': 0.70,
    'val_ratio': 0.20,
    'test_ratio': 0.10,
    'image_size': 240,            # Paper: 240x240
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,
    'class_names': ['Benign', 'Malignant', 'Normal'],
    'aug_factor_benign': 13,
    'aug_factor_malignant': 2,
    'aug_factor_normal': 3,
    'variants': ['b0', 'b1', 'b2', 'b3', 'b4'],
}

# ===================== REPRODUCIBILITY =====================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(config['seed'])

# ===================== DATA LOADING =====================
CLASS_FOLDERS = ["Bengin cases", "Malignant cases", "Normal cases"]
CLASS_MAP = {"Bengin cases": 0, "Malignant cases": 1, "Normal cases": 2}

def collect_all_images(data_dir):
    all_samples = []
    for cls_name, cls_id in CLASS_MAP.items():
        cls_dir = os.path.join(data_dir, cls_name)
        if not os.path.exists(cls_dir):
            for split in ['train', 'test', 'valid']:
                split_class = os.path.join(data_dir, split, cls_name)
                if os.path.exists(split_class):
                    cls_dir = split_class; break
        if os.path.exists(cls_dir):
            files = sorted([f for f in os.listdir(cls_dir)
                          if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            for f in files:
                all_samples.append((os.path.join(cls_dir, f), cls_id))
    return all_samples

def crop_lung_contour(image_np):
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY) if len(image_np.shape)==3 else image_np.copy()
    _, thresh = cv2.threshold(gray, LUNG_CROP_THRESH, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        pad = 5
        x, y = max(0, x-pad), max(0, y-pad)
        w = min(image_np.shape[1]-x, w+2*pad)
        h = min(image_np.shape[0]-y, h+2*pad)
        return image_np[y:y+h, x:x+w]
    return image_np

class LungCropTransform:
    def __call__(self, image):
        return Image.fromarray(crop_lung_contour(np.array(image)))

# ===================== DATASET =====================
class LungDataset(Dataset):
    def __init__(self, samples, transform, oversample_factors=None):
        self.transform = transform
        if oversample_factors:
            self.samples = []
            for path, label in samples:
                factor = oversample_factors.get(label, 1)
                self.samples.extend([(path, label)] * factor)
        else:
            self.samples = list(samples)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

# ===================== TRANSFORMS =====================
def get_transforms(image_size):
    train_tf = transforms.Compose([
        LungCropTransform(),
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2),
        transforms.RandomAffine(0, translate=(0.1,0.1), scale=(0.9,1.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    eval_tf = transforms.Compose([
        LungCropTransform(),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return train_tf, eval_tf

# ===================== DATA SPLIT (SEQUENTIAL - NO SHUFFLE) =====================
def prepare_sequential_data(data_dir, train_ratio=0.70, val_ratio=0.20, batch_size=32, image_size=240):
    """Sequential split: first 70% train, next 20% val, last 10% test. NO shuffle."""
    all_samples = collect_all_images(data_dir)

    print(f"Total images: {len(all_samples)}")
    for name, cid in CLASS_MAP.items():
        print(f"  {name}: {sum(1 for _,l in all_samples if l==cid)}")

    # Sequential split PER CLASS
    train_samples, val_samples, test_samples = [], [], []
    # --- v2: split via make_split(_SPLIT_MODE, SPLIT_RATIO, SEED) ---
    _seed_for_split = globals().get('_CUR_SEED', SEEDS[0])
    _per_class = {cid: [(p,l) for p,l in all_samples if l==cid]
                   for _, cid in CLASS_MAP.items()}
    train_samples, val_samples, test_samples = make_split(_per_class, _SPLIT_MODE, SPLIT_RATIO, _seed_for_split)
    # --- v2.1 empty-val fallback ---
    if len(val_samples) == 0:
        print('  [v2.1] val split empty — using test as val for monitoring.')
        val_samples = list(test_samples)

    print(f"\nAfter sequential 70/20/10 split:")
    print(f"  Train: {len(train_samples)} (first 70%)")
    print(f"  Val:   {len(val_samples)} (next 20%)")
    print(f"  Test:  {len(test_samples)} (last 10%)")
    for name, cid in CLASS_MAP.items():
        tr = sum(1 for _,l in train_samples if l==cid)
        va = sum(1 for _,l in val_samples if l==cid)
        te = sum(1 for _,l in test_samples if l==cid)
        print(f"    {name}: train={tr}, val={va}, test={te}")

    train_tf, eval_tf = get_transforms(image_size)
    aug_factors = {0: config['aug_factor_benign'], 1: config['aug_factor_malignant'],
                   2: config['aug_factor_normal']}

    train_ds = LungDataset(train_samples, train_tf, oversample_factors=aug_factors)
    val_ds = LungDataset(val_samples, eval_tf)
    test_ds = LungDataset(test_samples, eval_tf)
    print(f"Train after augmentation: {len(train_ds)}")

    kw = dict(num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    _lk = _loader_kwargs(globals().get('_CUR_SEED', SEEDS[0]))
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **kw, **_lk)
    val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kw)
    test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kw)
    # --- v2: val fallback when ratio yields 0 val samples ---
    try:
        if hasattr(val_ld, 'dataset') and len(val_ld.dataset) == 0:
            print('  [fallback] val split empty -> aliasing val_ld = test_ld (monitoring only)')
            val_ld = test_ld
    except Exception:
        pass

    return train_ld, val_ld, test_ld

# ===================== MODEL =====================
def build_lung_effnet(variant='b1', num_classes=3, dropout=0.5, pretrained=True):
    weights_map = {
        'b0': (tv_models.efficientnet_b0, tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        'b1': (tv_models.efficientnet_b1, tv_models.EfficientNet_B1_Weights.IMAGENET1K_V1),
        'b2': (tv_models.efficientnet_b2, tv_models.EfficientNet_B2_Weights.IMAGENET1K_V1),
        'b3': (tv_models.efficientnet_b3, tv_models.EfficientNet_B3_Weights.IMAGENET1K_V1),
        'b4': (tv_models.efficientnet_b4, tv_models.EfficientNet_B4_Weights.IMAGENET1K_V1),
    }
    model_fn, weights = weights_map[variant]
    model = model_fn(weights=weights if pretrained else None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    total = sum(p.numel() for p in model.parameters())
    print(f"  EfficientNet-{variant.upper()}: {total/1e6:.2f}M params")
    return model

# ===================== TRAINING =====================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device)
        out = model(imgs); loss = criterion(out, labels)
        loss_sum += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(F.softmax(out, dim=1).cpu().tolist())
    return loss_sum/total, correct/total, all_preds, all_labels, all_probs

def train_full(model, train_ld, val_ld, test_ld, name, device, epochs=30, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=config['lr_decay_factor'],
                                  patience=config['lr_patience'])
    best_val_acc, best_wts, no_improve = 0, None, 0
    early_stop_patience = 7

    print(f"\n{'='*60}\n Training: {name} | {epochs} epochs | LR={lr}\n{'='*60}")
    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_ld, optimizer, criterion, device)
        vl_loss, vl_acc, vl_preds, vl_labels, _ = evaluate(model, val_ld, criterion, device)
        vl_f1 = f1_score(vl_labels, vl_preds, average='macro', zero_division=0)
        scheduler.step(vl_acc)
        cur_lr = optimizer.param_groups[0]['lr']

        marker = ''
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_wts = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' ★'
        else:
            no_improve += 1

        print(f"  [{epoch+1:3d}/{epochs}] loss={tr_loss:.4f}/{vl_loss:.4f} "
              f"acc={tr_acc:.4f}/{vl_acc:.4f} f1={vl_f1:.4f} "
              f"lr={cur_lr:.1e} ({time.time()-t0:.1f}s){marker}")

        wandb.log({"epoch": epoch+1, "train/loss": tr_loss, "train/acc": tr_acc*100,
                    "val/loss": vl_loss, "val/acc": vl_acc*100, "val/f1": vl_f1*100,
                    "best_val_acc": best_val_acc*100, "learning_rate": cur_lr})

        if no_improve >= early_stop_patience:
            print(f"  ⏹ Early stop at epoch {epoch+1} (no improvement for {early_stop_patience} epochs)")
            break

    # Load best & test
    if best_wts: model.load_state_dict(best_wts)
    _, test_acc, test_preds, test_labels, test_probs = evaluate(model, test_ld, criterion, device)
    test_prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    test_rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    test_f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    try:
        test_auc = _safe_auc(test_labels, test_probs, multi_class='ovr', average='macro')
    except Exception:
        test_auc = 0.0
    return model, {'name': name, 'test_acc': test_acc, 'test_preds': test_preds,
                   'test_labels': test_labels, 'test_probs': test_probs,
                   'test_prec': test_prec, 'test_rec': test_rec,
                   'test_f1': test_f1, 'test_auc': test_auc,
                   'params': sum(p.numel() for p in model.parameters())}

# ===================== EVALUATION =====================
def full_evaluation(results, class_names):
    preds, labels, probs = results['test_preds'], results['test_labels'], results['test_probs']
    print(f"\n{'='*60}\n TEST RESULTS — {results['name']}\n{'='*60}")
    print(f"Accuracy:  {results['test_acc']*100:.2f}%")
    print(f"Parameters: {results['params']/1e6:.2f}M\n")
    report = classification_report(labels, preds, target_names=class_names, zero_division=0)
    print(report)

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
                yticklabels=class_names, ax=ax, annot_kws={'size': 16})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f"{results['name']} — Confusion Matrix")
    plt.tight_layout()
    wandb.log({f"confusion/{results['name']}": wandb.Image(fig)})
    plt.show(); plt.close()

    # Log to wandb
    try:
        auc_val = _safe_auc(labels, probs, multi_class='ovr', average='macro')
    except:
        auc_val = 0
    wandb.log({f"test/{results['name']}_acc": results['test_acc']*100,
               f"test/{results['name']}_auc": auc_val*100})

# ===================== XAI: Grad-CAM =====================
def get_gradcam(model, image_tensor, target_class, device):
    model.eval()
    activations, gradients = {}, {}
    def fwd_hook(m, i, o): activations['v'] = o.detach()
    def bwd_hook(m, gi, go): gradients['v'] = go[0].detach()

    target_layer = model.features[-1]  # last conv block
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    inp = image_tensor.unsqueeze(0).to(device).requires_grad_(True)
    out = model(inp); model.zero_grad()
    out[0, target_class].backward()

    w = gradients['v'].mean(dim=[2,3], keepdim=True)
    cam = F.relu((w * activations['v']).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(config['image_size'],config['image_size']),
                        mode='bilinear', align_corners=False)
    cam = cam.squeeze().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    h1.remove(); h2.remove()
    return cam

def visualize_xai(model, test_loader, device):
    model.eval()
    samples = {}
    for imgs, labels in test_loader:
        for i in range(imgs.size(0)):
            lbl = labels[i].item() if isinstance(labels[i], torch.Tensor) else labels[i]
            if lbl not in samples: samples[lbl] = imgs[i]
        if len(samples) >= 3: break

    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    for row, (lbl, img_t) in enumerate(sorted(samples.items())):
        with torch.enable_grad():
            cam = get_gradcam(model, img_t, lbl, device)
        img_np = img_t.permute(1,2,0).cpu().numpy()
        img_np = img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
        img_np = np.clip(img_np, 0, 1)

        axes[row,0].imshow(img_np); axes[row,0].set_title(f'Original\nTrue: {config["class_names"][lbl]}'); axes[row,0].axis('off')
        axes[row,1].imshow(cam, cmap='jet'); axes[row,1].set_title('Grad-CAM'); axes[row,1].axis('off')
        overlay = np.clip(0.5*img_np + 0.5*plt.cm.jet(cam)[:,:,:3], 0, 1)
        axes[row,2].imshow(overlay); axes[row,2].set_title('Overlay'); axes[row,2].axis('off')

    fig.suptitle('XAI: Grad-CAM (EfficientNet Last Conv)', fontsize=15, fontweight='bold')
    plt.tight_layout()
    wandb.log({"xai/gradcam": wandb.Image(fig)})
    plt.show(); plt.close()
    print("\n✅ XAI visualizations generated.")

# ===================== MAIN EXECUTION =====================
if __name__ == '__main__':
    _ALL_SEED_RESULTS = []
    for _SPLIT_MODE in SPLIT_MODES:
        for _CUR_SEED in SEEDS:
            print('\n' + '='*70)
            print(f' SEED = {_CUR_SEED} | _SPLIT_MODE = {_SPLIT_MODE} | RATIO = {SPLIT_RATIO}')
            print('='*70)
            set_seed(_CUR_SEED)
            _wandb_login_safe()

            _wandb_init_safe(
                project="lung_effnet_reproduction",
                name=f"effnet_{_SPLIT_MODE}_seed{_CUR_SEED}",
                config=config,
                tags=[f"split:{_SPLIT_MODE}", f"seed:{_CUR_SEED}", "v2"],
                notes="Lung-EffNet honest benchmark: 70/20/10 sequential split (no shuffle)"
            )

            print("="*60)
            print("Lung-EffNet — Sequential 70/20/10 Split (No Shuffle)")
            print("="*60)
            train_ld, val_ld, test_ld = prepare_sequential_data(
                config['data_dir'], train_ratio=config['train_ratio'],
                val_ratio=config['val_ratio'],
                batch_size=config['batch_size'], image_size=config['image_size'])

            wandb.log({"dataset/train_samples": len(train_ld.dataset),
                       "dataset/val_samples": len(val_ld.dataset),
                       "dataset/test_samples": len(test_ld.dataset),
                       "dataset/split_method": f"{_SPLIT_MODE} ratio={SPLIT_RATIO}"})

            # Paper results for comparison
            paper_results = {
                'B0': {'acc': 93.67, 'f1': 88.18}, 'B1': {'acc': 99.10, 'f1': 98.16},
                'B2': {'acc': 99.10, 'f1': 98.16}, 'B3': {'acc': 98.19, 'f1': 96.19},
                'B4': {'acc': 97.29, 'f1': 94.29},
            }

            all_results = {}
            best_model = None
            best_acc = 0

            for variant in config['variants']:
                print(f"\n{'#'*60}")
                print(f" EfficientNet-{variant.upper()} | SEQUENTIAL SPLIT")
                print(f"{'#'*60}")
                model = build_lung_effnet(variant, dropout=config['dropout'])
                model, results = train_full(model, train_ld, val_ld, test_ld,
                                             f'EffNet-{variant.upper()}', config['device'],
                                             epochs=config['epochs'], lr=config['learning_rate'])
                full_evaluation(results, config['class_names'])
                all_results[variant.upper()] = results
                if results['test_acc'] > best_acc:
                    best_acc = results['test_acc']; best_model = model
                del model; torch.cuda.empty_cache(); gc.collect()

            # Comparison table
            print(f"\n{'='*75}")
            print(f" COMPARISON: Paper (Leaky) vs Sequential (Honest)")
            print(f"{'='*75}")
            print(f"{'Model':<12} {'Paper Acc':>10} {'Seq Acc':>10} {'Δ Acc':>8}")
            print("-"*45)
            for v in ['B0','B1','B2','B3','B4']:
                if v in all_results:
                    our = all_results[v]['test_acc']*100
                    paper = paper_results[v]['acc']
                    print(f"EffNet-{v:<5} {paper:>9.2f}% {our:>9.2f}% {our-paper:>+7.2f}%")

            # XAI on best model
            if best_model:
                print("\nGenerating XAI on best model...")
                visualize_xai(best_model, test_ld, config['device'])
            # --- v2: collect per-run metrics ---
            try:
                _entry = {'split_mode': _SPLIT_MODE,
                          'split_ratio': str(SPLIT_RATIO),
                          'seed': _CUR_SEED,
                          'model': '__main__'}
                # pull metrics from local scope (test_acc, results dict, etc.)
                _src = None
                try:
                    _src = all_results  # dict of {variant: results}
                except NameError:
                    pass
                if isinstance(_src, dict) and _src:
                    for _v, _r in _src.items():
                        _row = dict(_entry); _row['model'] = _v
                        _row['accuracy']  = float(_r.get('test_acc',  0)) * 100
                        _row['precision'] = float(_r.get('test_prec', _r.get('precision', 0))) * 100
                        _row['recall']    = float(_r.get('test_rec',  _r.get('recall',    0))) * 100
                        _row['f1']        = float(_r.get('test_f1',   _r.get('f1',        0))) * 100
                        _row['auc']       = float(_r.get('test_auc',  _r.get('auc',       0))) * 100
                        _ALL_SEED_RESULTS.append(_row)
                else:
                    try:
                        _entry['accuracy'] = float(test_acc) * 100
                    except Exception: pass
                    try: _entry['auc'] = float(auc) * 100
                    except Exception: pass
                    _ALL_SEED_RESULTS.append(_entry)
            except Exception as _re:
                print(f'[v2] result-collect skipped: {_re}')


            wandb.finish()
            print("Done!")

    # --- v2 aggregate across modes x seeds ---
    try:
        if _ALL_SEED_RESULTS:
            _df = pd.DataFrame(_ALL_SEED_RESULTS)
            out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
            _csv = f"{out_dir}/summary_{'_'.join(SPLIT_MODES)}.csv"
            _df.to_csv(_csv, index=False)
            print("\nSeed-aggregated summary:"); print(_df)
            try:
                num_cols = [c for c in ['accuracy','precision','recall','f1','auc'] if c in _df.columns]
                agg = _df.groupby('split_mode')[num_cols].agg(['mean','std'])
                print("\nMean +/- Std per split_mode:"); print(agg)
            except Exception as _ag:
                print(f'[agg] groupby skipped: {_ag}')
            print(f'Saved -> {_csv}')
    except Exception as _e:
        print(f'[v2] aggregate skipped: {_e}')